In [3]:
!pip install -q sentence-transformers
!pip install -q vaderSentiment
!pip install -q pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.3 MB/s eta 0:00:00


In [4]:
import os
import json
import re
import nltk
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from collections import defaultdict

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [5]:
PROJECT_ROOT = "/content/drive/MyDrive/expo"

DATA_DIR = PROJECT_ROOT
ARTIFACT_DIR = f"{PROJECT_ROOT}/artifacts"

os.makedirs(
    ARTIFACT_DIR,
    exist_ok=True
)

CELL 4 — Load Datasets

In [6]:
reviews = pd.read_json(
    f"{DATA_DIR}/hotel_reviews.json"
)

profiles = pd.read_json(
    f"{DATA_DIR}/user_profiles.json"
)

print(reviews.shape)
print(profiles.shape)

reviews.head()

(50000, 9)
(50, 2)


,review_id,hotel_id,hotel_name,hotel_category,rating,review_date,review_text,verified,traveler_type
0,R31838,H051,"Casa Harbour, Istanbul",5-star,4,2024-11-23,Second time staying at this property. Pure lux...,False,NaN
1,R25484,H037,"Hotel Lantern, Barcelona",3-star,4,2025-05-19,The kids loved the pool and the staff were won...,True,NaN
2,R46705,H089,"The Regent Heights, Lisbon",5-star,3,2025-09-21,Second time staying at this property. You can'...,True,solo
3,R35771,H062,"The Grand Heights, Tokyo",3-star,2,2024-09-13,Breakfast was overpriced at $45 and surprising...,True,NaN
4,R56675,H114,"Casa Solana, Rome",5-star,5,2024-04-28,Plenty for children to do and the family suite...,False,family


CELL 5 — Aspect Taxonomy

In [7]:
ASPECTS = [
    "safety",
    "location",
    "culture",
    "business",
    "family",
    "wellness",
    "service",
    "cleanliness",
    "value",
    "accessibility",
    "nightlife",
    "beach"
]

CELL 6 — Aspect Descriptions

In [8]:
ASPECT_DESCRIPTIONS = {

    "safety":
        "safe secure dangerous neighborhood crime lighting alone at night",

    "location":
        "central walkable attractions transport downtown convenient nearby",

    "culture":
        "local culture authentic neighborhood local food local markets tourist",

    "business":
        "wifi internet remote work business desk meeting conference",

    "family":
        "kids children family toddler family room",

    "wellness":
        "spa sauna massage yoga relaxation wellness",

    "service":
        "staff concierge service helpful rude reception hospitality",

    "cleanliness":
        "clean dirty spotless bathroom housekeeping hygiene",

    "value":
        "cheap expensive overpriced value budget affordable",

    "accessibility":
        "wheelchair elevator ramps accessible mobility disability",

    "nightlife":
        "bars clubs nightlife pubs entertainment",

    "beach":
        "beach ocean sea shoreline waterfront"
}

CELL 7 — Updated Keywords

In [9]:
ASPECT_KEYWORDS = {

    "business": [
        "wifi","internet","desk","workspace",
        "meeting","conference","remote work",
        "video calls","business travel",
        "work trip","reliable internet"
    ],

    "wellness": [
        "spa","massage","sauna",
        "relaxation","wellness",
        "treatments","steam room",
        "therapy","refresh"
    ],

    "family": [
        "kids","children","child",
        "toddler","family room",
        "family suite","play area",
        "kids club","baby","crib"
    ],

    "cleanliness": [
        "clean","dirty","spotless",
        "bathroom","housekeeping",
        "hygiene","hair","dust",
        "stains","fresh"
    ],

    "service": [
        "staff","service","concierge",
        "reception","front desk",
        "helpful","friendly",
        "attentive","professional",
        "hospitality","check in",
        "check-in","check out",
        "check-out","rude",
        "slow service",
        "personal service"
    ],

    "safety": [
        "safe","unsafe","secure",
        "security","crime",
        "sketchy","well lit",
        "lighting","alone at night"
    ],

    "nightlife": [
        "bars","clubs","nightlife",
        "pubs","cocktails",
        "late night","party"
    ],

    "location": [
        "location","central",
        "walkable","downtown",
        "transport","subway",
        "metro","station",
        "airport","near",
        "nearby","close to",
        "minutes away",
        "walking distance",
        "conveniently located"
    ],

    "culture": [
        "authentic","culture",
        "local","local markets",
        "local dishes",
        "tourist bubble",
        "neighborhood",
        "touristy"
    ],

    "accessibility": [
        "wheelchair","accessible",
        "accessibility","mobility",
        "reduced mobility",
        "stairs","step-free",
        "step heavy","step-heavy",
        "lift","elevator",
        "ramp","walker",
        "crutches","disabled",
        "barrier free"
    ],

    "beach": [
        "beach","ocean","sea",
        "shore","shoreline",
        "waterfront","beachfront"
    ],

    "value": [
        "cheap","expensive",
        "overpriced","budget",
        "affordable","value",
        "worth","worth it",
        "price","pricing",
        "good deal","great deal",
        "cost","reasonable",
        "premium"
    ]
}

CELL 8 — Load Models

In [10]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

sentiment_model = (
    SentimentIntensityAnalyzer()
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

CELL 9 — Create Aspect Embeddings

In [11]:
aspect_embeddings = model.encode(
    list(
        ASPECT_DESCRIPTIONS.values()
    ),
    normalize_embeddings=True
)

print(
    aspect_embeddings.shape
)

(12, 384)


CELL 10 — Sentence Splitter

In [12]:
def split_sentences(text):

    return nltk.sent_tokenize(
        str(text)
    )

CELL 11 — Precompute Unique Sentence Embeddings

In [13]:
unique_sentences = set()

for review in reviews["review_text"]:
    unique_sentences.update(
        split_sentences(review)
    )

unique_sentences = list(
    unique_sentences
)

print(
    "Unique Sentences:",
    len(unique_sentences)
)

Unique Sentences: 87


CELL 12 — Batch Encode Sentences

In [14]:
sentence_embeddings = model.encode(
    unique_sentences,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

CELL 13 — Create Lookup Dictionary

In [15]:
sentence_embedding_lookup = {
    sentence: embedding
    for sentence, embedding
    in zip(
        unique_sentences,
        sentence_embeddings
    )
}

CELL 14 — Keyword Scoring

In [16]:
def keyword_score(sentence):

    scores = defaultdict(float)

    sentence_lower = sentence.lower()

    for aspect, words in ASPECT_KEYWORDS.items():

        for word in words:

            pattern = rf"\b{re.escape(word.lower())}\b"

            if re.search(
                pattern,
                sentence_lower
            ):
                scores[aspect] += 1

    return scores

CELL 15 — Embedding Scoring

In [17]:
def embedding_score(sentence):

    emb = (
        sentence_embedding_lookup[
            sentence
        ]
        .reshape(1, -1)
    )

    sims = cosine_similarity(
        emb,
        aspect_embeddings
    )[0]

    return {
        aspect: float(sim)
        for aspect, sim
        in zip(
            ASPECTS,
            sims
        )
    }

CELL 16 — Hybrid Score

In [18]:
def hybrid_score(sentence):

    kw = keyword_score(sentence)
    emb = embedding_score(sentence)

    final_scores = {}

    for aspect in ASPECTS:

        keyword_component = min(
            kw.get(aspect,0),
            1
        )

        final_scores[aspect] = (
            0.55 * keyword_component
            +
            0.45 * emb.get(aspect,0)
        )

    return final_scores

CELL 17 — Sentiment

In [19]:
def get_sentiment(text):

    return sentiment_model\
        .polarity_scores(
            text
        )["compound"]

CELL 18 — Review Processing

In [20]:
def process_review(review):

    results = []

    for sentence in split_sentences(
        review
    ):

        scores = hybrid_score(
            sentence
        )

        sentiment = get_sentiment(
            sentence
        )

        for aspect, score in scores.items():

            if score >= 0.50:

                results.append({
                    "aspect": aspect,
                    "confidence": round(
                        score,
                        4
                    ),
                    "sentiment": round(
                        sentiment,
                        4
                    ),
                    "sentence": sentence
                })

    return results

CELL 19 — Calibration Check

In [21]:
sample_reviews = reviews.sample(
    10,
    random_state=42
)

for review in sample_reviews[
    "review_text"
]:

    print("="*100)

    print(review)

    print(
        process_review(review)
    )

    print()

We had high hopes for this stay. The level of refinement and personal service was world-class. Loved that it sat in a genuine local neighborhood, not a tourist bubble.
[{'aspect': 'service', 'confidence': 0.688, 'sentiment': 0.0, 'sentence': 'The level of refinement and personal service was world-class.'}, {'aspect': 'culture', 'confidence': 0.7793, 'sentiment': 0.5994, 'sentence': 'Loved that it sat in a genuine local neighborhood, not a tourist bubble.'}]

Wonderful wellness facilities; the sauna and treatments left us completely refreshed. Overall a solid choice.
[{'aspect': 'wellness', 'confidence': 0.8225, 'sentiment': 0.765, 'sentence': 'Wonderful wellness facilities; the sauna and treatments left us completely refreshed.'}]

Second time staying at this property. Slept beautifully — very quiet despite being central. On-site restaurant was a highlight — beautifully prepared local dishes. Decent for one night, not for longer.
[{'aspect': 'location', 'confidence': 0.6139, 'sentiment

CELL 20 — Full Extraction

In [22]:
all_results = []

for _, row in tqdm(
    reviews.iterrows(),
    total=len(reviews)
):

    extracted = process_review(
        row["review_text"]
    )

    for item in extracted:

        item["review_id"] = (
            row["review_id"]
        )

        item["hotel_id"] = (
            row["hotel_id"]
        )

        all_results.append(
            item
        )

aspect_df = pd.DataFrame(
    all_results
)

aspect_df.head()

  0%|          | 0/50000 [00:00<?, ?it/s]

,aspect,confidence,sentiment,sentence,review_id,hotel_id
0,service,0.6911,0.0000,Pure luxury from check-in to checkout — utterl...,R31838,H051
1,accessibility,0.7030,0.0000,"Fully step-free with a spacious, well-designed...",R31838,H051
2,cleanliness,0.8005,-0.3089,Room wasn't clean on arrival — found hair in t...,R31838,H051
3,family,0.7460,0.8225,The kids loved the pool and the staff were won...,R25484,H037
4,service,0.6490,0.8225,The kids loved the pool and the staff were won...,R25484,H037


CELL 21 — Diagnostics

In [23]:
print(
    aspect_df["aspect"]
    .value_counts()
)

print()

print(
    aspect_df["sentiment"]
    .describe()
)

print()

print(
    aspect_df.groupby(
        "review_id"
    ).size().describe()
)

aspect
service          15404
location          9940
business          9798
culture           9644
value             7390
wellness          6769
cleanliness       6535
accessibility     5927
beach             5261
nightlife         4217
family            4216
safety            3237
Name: count, dtype: int64

count    88338.000000
mean         0.255993
std          0.452395
min         -0.796000
25%          0.000000
50%          0.296000
75%          0.718400
max          0.840200
Name: sentiment, dtype: float64

count    47197.000000
mean         1.871687
std          0.821244
min          1.000000
25%          1.000000
50%          2.000000
75%          2.000000
max          6.000000
dtype: float64


CELL 22 — Save Artifacts

In [24]:
aspect_df.to_parquet(
    f"{ARTIFACT_DIR}/review_aspects.parquet",
    index=False
)

np.save(
    f"{ARTIFACT_DIR}/aspect_embeddings.npy",
    aspect_embeddings
)

with open(
    f"{ARTIFACT_DIR}/aspect_descriptions.json",
    "w"
) as f:

    json.dump(
        ASPECT_DESCRIPTIONS,
        f,
        indent=4
    )

with open(
    f"{ARTIFACT_DIR}/aspect_keywords.json",
    "w"
) as f:

    json.dump(
        ASPECT_KEYWORDS,
        f,
        indent=4
    )

print(
    "\nArtifacts Saved Successfully\n"
)

print(
    os.listdir(
        ARTIFACT_DIR
    )
)


Artifacts Saved Successfully

['review_aspects.parquet', 'aspect_embeddings.npy', 'aspect_descriptions.json', 'aspect_keywords.json', 'reviews_clean.parquet', 'profiles_clean.parquet', 'hotel_categories.json', 'monthly_hotel_stats.parquet', 'hotel_stats.parquet', 'dataset_metadata.json', 'top_words.json', 'top_bigrams.json', 'hotel_review_counts.json']
